# Working with Bill Objects

The `Bill` class is the foundation of the Sinking Fund library. It represents financial obligations that you need to plan for, whether they're one-time expenses or recurring payments.

This notebook explores:
- Creating one-time and recurring bills programmatically
- Generating bill instances for specific date ranges
- Calendar-aware date calculations (month-end adjustments, leap years)
- Understanding bill lifecycle and occurrence patterns


## Import Required Modules

Let's start by importing the necessary classes:


In [ ]:
from datetime import date
from decimal import Decimal
from sinkingfund import Bill, BillInstance


## Creating One-Time Bills

A one-time bill represents a single payment obligation with a specific due date. This is perfect for expenses like property tax, annual fees, or one-off purchases.


In [ ]:
# Create a one-time property tax bill.
property_tax = Bill(
    bill_id="prop_tax_2025",
    service="Property Tax 2025",
    amount_due=3600.00,
    recurring=False,
    due_date=date(2025, 11, 1)
)

print(f"Bill ID: {property_tax.bill_id}")
print(f"Service: {property_tax.service}")
print(f"Amount Due: ${property_tax.amount_due}")
print(f"Recurring: {property_tax.recurring}")
print(f"Due Date: {property_tax.start_date}")  # For non-recurring bills, start_date is the due date
print(f"Occurrences: {property_tax.occurrences}")  # Always 1 for one-time bills


## Creating Recurring Bills

Recurring bills repeat over time according to a schedule. You can specify the frequency (daily, weekly, monthly, quarterly, annual) and the interval between occurrences.


In [ ]:
# Monthly bill (every month).
monthly_insurance = Bill(
    bill_id="car_insurance",
    service="Car Insurance",
    amount_due=750.00,
    recurring=True,
    start_date=date(2025, 4, 24),
    frequency="monthly",
    interval=6  # Every 6 months (semi-annual, but uses monthly frequency)
)

print(f"Bill ID: {monthly_insurance.bill_id}")
print(f"Service: {monthly_insurance.service}")
print(f"Frequency: {monthly_insurance.frequency}")
print(f"Interval: {monthly_insurance.interval} months")
print(f"Start Date: {monthly_insurance.start_date}")
print(f"First occurrence: {monthly_insurance.start_date}")


### Controlling Recurrence with Occurrences or End Date

You can limit recurring bills in two ways:
1. **occurrences**: Specify how many times the bill occurs
2. **end_date**: Specify when the bill stops recurring

Let's see both approaches:


In [ ]:
# Bill that occurs exactly 12 times (12 months).
yearly_contract = Bill(
    bill_id="yearly_subscription",
    service="Yearly Subscription",
    amount_due=120.00,
    recurring=True,
    start_date=date(2025, 1, 1),
    frequency="monthly",
    interval=1,
    occurrences=12  # Exactly 12 occurrences
)

print(f"Occurrences: {yearly_contract.occurrences}")
print(f"Start Date: {yearly_contract.start_date}")
print(f"End Date: {yearly_contract.end_date}")  # Automatically calculated

# Bill that runs until a specific end date.
quarterly_tax = Bill(
    bill_id="quarterly_tax",
    service="Quarterly Tax Payments",
    amount_due=1500.00,
    recurring=True,
    start_date=date(2025, 4, 15),
    frequency="quarterly",
    interval=1,
    end_date=date(2026, 4, 15)  # Runs for one year
)

print(f"\nQuarterly Tax Bill:")
print(f"Start Date: {quarterly_tax.start_date}")
print(f"End Date: {quarterly_tax.end_date}")
print(f"Occurrences: {quarterly_tax.occurrences}")  # Automatically calculated


## Generating Bill Instances for Date Ranges

A `Bill` defines the pattern, but `BillInstance` objects represent specific occurrences. The `instances_in_range()` method generates all bill instances within a specified date range.


In [ ]:
# Create a monthly bill.
monthly_bill = Bill(
    bill_id="monthly_utility",
    service="Monthly Utility Bill",
    amount_due=150.00,
    recurring=True,
    start_date=date(2025, 1, 15),
    frequency="monthly",
    interval=1,
    occurrences=12
)

# Get all instances for 2025.
instances = monthly_bill.instances_in_range(
    start_reference=date(2025, 1, 1),
    end_reference=date(2025, 12, 31)
)

print(f"Number of instances in 2025: {len(instances)}")
print("\nInstances:")
for i, instance in enumerate(instances, 1):
    print(f"  {i}. {instance.service} - ${instance.amount_due} due {instance.due_date}")


### Getting the Next Instance

The `next_instance()` method finds the next occurrence of a bill relative to a reference date. This is useful for finding upcoming payments.


In [ ]:
# Reference date: March 10, 2025.
reference = date(2025, 3, 10)

# Get next instance (exclusive - must be after reference date).
next_bill = monthly_bill.next_instance(reference_date=reference, inclusive=False)

print(f"Reference date: {reference}")
print(f"Next bill after {reference}:")
if next_bill:
    print(f"  Service: {next_bill.service}")
    print(f"  Amount: ${next_bill.amount_due}")
    print(f"  Due Date: {next_bill.due_date}")

# Get next instance (inclusive - includes reference date if it's a due date).
next_bill_inclusive = monthly_bill.next_instance(reference_date=reference, inclusive=True)

print(f"\nNext bill on or after {reference} (inclusive):")
if next_bill_inclusive:
    print(f"  Due Date: {next_bill_inclusive.due_date}")


## Calendar-Aware Date Calculations

The Bill class handles calendar complexities automatically. This is especially important for monthly bills where the day of the month matters.

### Month-End Adjustments

Bills due on the 31st (or other late days) automatically adjust to the last day of shorter months:


In [ ]:
# Monthly bill due on the 31st.
bill_31st = Bill(
    bill_id="monthly_rent",
    service="Monthly Rent",
    amount_due=1200.00,
    recurring=True,
    start_date=date(2025, 1, 31),  # January 31st
    frequency="monthly",
    interval=1,
    occurrences=12
)

# Get instances for first 6 months.
instances = bill_31st.instances_in_range(
    start_reference=date(2025, 1, 1),
    end_reference=date(2025, 6, 30)
)

print("Monthly rent due on the 31st:")
for instance in instances:
    print(f"  {instance.due_date.strftime('%B %d, %Y')} - ${instance.amount_due}")
    print(f"    (Month has {instance.due_date.day} days)")

# Notice how February adjusts to the 28th (or 29th in leap years)
# and April adjusts to the 30th!


### Leap Year Handling

Bills starting on February 29th (leap day) automatically adjust to February 28th in non-leap years:


In [ ]:
# Bill starting in a leap year (2024).
leap_year_bill = Bill(
    bill_id="leap_year_example",
    service="Leap Year Example",
    amount_due=100.00,
    recurring=True,
    start_date=date(2024, 2, 29),  # Leap day!
    frequency="annual",
    interval=1,
    occurrences=5
)

# Get instances spanning multiple years.
instances = leap_year_bill.instances_in_range(
    start_reference=date(2024, 1, 1),
    end_reference=date(2029, 12, 31)
)

print("Annual bill starting on February 29, 2024 (leap day):")
for instance in instances:
    year = instance.due_date.year
    is_leap = (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)
    print(f"  {instance.due_date} ({'Leap year' if is_leap else 'Non-leap year'})")

# Notice how non-leap years get February 28th instead!


## Different Frequency Types

The library supports various frequencies. Let's explore them:


In [ ]:
# Daily bill.
daily_bill = Bill(
    bill_id="daily_expense",
    service="Daily Parking",
    amount_due=5.00,
    recurring=True,
    start_date=date(2025, 1, 1),
    frequency="daily",
    interval=1,
    occurrences=7  # One week
)

# Weekly bill.
weekly_bill = Bill(
    bill_id="weekly_groceries",
    service="Weekly Groceries",
    amount_due=100.00,
    recurring=True,
    start_date=date(2025, 1, 6),  # Monday
    frequency="weekly",
    interval=1,
    occurrences=4  # One month
)

# Quarterly bill.
quarterly_bill = Bill(
    bill_id="quarterly_tax",
    service="Quarterly Tax",
    amount_due=500.00,
    recurring=True,
    start_date=date(2025, 4, 15),
    frequency="quarterly",
    interval=1,
    occurrences=4  # One year
)

# Annual bill.
annual_bill = Bill(
    bill_id="annual_fee",
    service="Annual Membership",
    amount_due=250.00,
    recurring=True,
    start_date=date(2025, 1, 1),
    frequency="annual",
    interval=1,
    occurrences=3  # Three years
)

print("Daily bill (first week of 2025):")
daily_instances = daily_bill.instances_in_range(date(2025, 1, 1), date(2025, 1, 7))
for instance in daily_instances:
    print(f"  {instance.due_date.strftime('%A, %B %d')} - ${instance.amount_due}")

print("\nWeekly bill (January 2025):")
weekly_instances = weekly_bill.instances_in_range(date(2025, 1, 1), date(2025, 1, 31))
for instance in weekly_instances:
    print(f"  {instance.due_date.strftime('%A, %B %d')} - ${instance.amount_due}")

print("\nQuarterly bill (2025):")
quarterly_instances = quarterly_bill.instances_in_range(date(2025, 1, 1), date(2025, 12, 31))
for instance in quarterly_instances:
    print(f"  {instance.due_date.strftime('%B %d, %Y')} - ${instance.amount_due}")

print("\nAnnual bill (2025-2027):")
annual_instances = annual_bill.instances_in_range(date(2025, 1, 1), date(2027, 12, 31))
for instance in annual_instances:
    print(f"  {instance.due_date.strftime('%B %d, %Y')} - ${instance.amount_due}")


In [ ]:
# Bi-weekly bill (every 2 weeks).
biweekly_bill = Bill(
    bill_id="biweekly_paycheck",
    service="Bi-weekly Paycheck",
    amount_due=2000.00,
    recurring=True,
    start_date=date(2025, 1, 3),  # First Friday
    frequency="weekly",
    interval=2,  # Every 2 weeks
    occurrences=26  # One year
)

# Bi-monthly bill (every 2 months).
bimonthly_bill = Bill(
    bill_id="bimonthly_maintenance",
    service="Bi-monthly Maintenance",
    amount_due=150.00,
    recurring=True,
    start_date=date(2025, 1, 15),
    frequency="monthly",
    interval=2,  # Every 2 months
    occurrences=6  # One year
)

print("Bi-weekly bill (first quarter 2025):")
biweekly_instances = biweekly_bill.instances_in_range(date(2025, 1, 1), date(2025, 3, 31))
for instance in biweekly_instances[:6]:  # Show first 6
    print(f"  {instance.due_date.strftime('%B %d, %Y')} - ${instance.amount_due}")

print("\nBi-monthly bill (2025):")
bimonthly_instances = bimonthly_bill.instances_in_range(date(2025, 1, 1), date(2025, 12, 31))
for instance in bimonthly_instances:
    print(f"  {instance.due_date.strftime('%B %d, %Y')} - ${instance.amount_due}")


## Summary

**Key Takeaways:**

1. **One-Time Bills**: Use `recurring=False` and specify a `due_date`. Perfect for single expenses.

2. **Recurring Bills**: Use `recurring=True` with `start_date`, `frequency`, and optionally `interval`, `occurrences`, or `end_date`.

3. **Bill Instances**: Use `instances_in_range()` to get all occurrences in a date range, or `next_instance()` to find the next payment.

4. **Calendar Intelligence**: The library automatically handles:
   - Month-end adjustments (31st → last day of shorter months)
   - Leap years (Feb 29 → Feb 28 in non-leap years)
   - Various frequencies (daily, weekly, monthly, quarterly, annual)

5. **Flexible Scheduling**: Use `interval` to create bills that occur less frequently than the base frequency.

**Next Steps:**
- Learn how bills become envelopes in the Envelope Lifecycle Management notebook
- Explore how bill instances drive cash flow schedules
- See bills in action with the SinkingFund workflow examples
